In [ ]:
# ==============================================================================
# 1. 환경 설정
# ==============================================================================

# 구글 드라이브 마운트
print("\n▼▼▼ 구글 드라이브 마운트 ▼▼▼")
from google.colab import drive, files
drive.mount('/content/drive')

# requirements.txt 업로드
uploaded = files.upload()
!pip install -q -r requirements.txt



▼▼▼ 구글 드라이브 마운트 ▼▼▼
Mounted at /content/drive


Saving requirements.txt to requirements.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 8.1 MB/s eta 0:00:00
  

In [ ]:
# ======================================================================
# 2. 데이터셋 준비 및 변환
# ======================================================================
import json, os, glob, shutil, yaml
from tqdm import tqdm

# ★ 데이터 경로 설정 (환경에 맞게 수정)
DRIVE_ROOT = '/content/drive/MyDrive/53.대기오염 배출원 공간 분포 데이터/3.개방데이터/1.데이터'

# 원본 데이터 경로
TRAIN_IMG_DIR_GDRIVE = os.path.join(DRIVE_ROOT, 'Training/01.원천데이터/TS_KS')
TRAIN_JSON_DIR_GDRIVE = os.path.join(DRIVE_ROOT, 'Training/02.라벨링데이터/TL_KS_BBOX')
VAL_IMG_DIR_GDRIVE   = os.path.join(DRIVE_ROOT, 'Validation/01.원천데이터/VS_KS')
VAL_JSON_DIR_GDRIVE  = os.path.join(DRIVE_ROOT, 'Validation/02.라벨링데이터/VL_KS_BBOX')

# YOLOv8용 데이터셋 저장 위치
DATASET_ROOT_LOCAL = '/content/drive/MyDrive/Colab Notebooks'

def setup_dataset_structure(root_path):
    """YOLO 학습에 필요한 폴더 구조를 생성"""
    os.makedirs(os.path.join(root_path, 'images/train'), exist_ok=True)
    os.makedirs(os.path.join(root_path, 'labels/train'), exist_ok=True)
    os.makedirs(os.path.join(root_path, 'images/val'),   exist_ok=True)
    os.makedirs(os.path.join(root_path, 'labels/val'),   exist_ok=True)
    print(f"Dataset structure created at: {root_path}")

def _find_image_any_ext(img_dir, base):
    """확장자 대소문자/형식(.jpg/.jpeg/.png/...)을 모두 허용해 이미지 탐색"""
    exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff',
            '.JPG', '.JPEG', '.PNG', '.BMP', '.TIF', '.TIFF')
    for ext in exts:
        p = os.path.join(img_dir, base + ext)
        if os.path.exists(p):
            return p, ext
    return None, None

def convert_and_copy_data(gdrive_json_dir, gdrive_img_dir, local_dest_dir,
                          set_type, resume=True, overwrite=False):
    """
    JSON → YOLO txt 변환 및 이미지 복사.
    resume=True 면 이미 만들어진 (라벨+이미지) 쌍은 건너뜀.
    overwrite=True 면 항상 다시 생성/복사.
    """
    print(f"\nProcessing {set_type} data...")
    json_files = glob.glob(os.path.join(gdrive_json_dir, '*.json'))

    skipped, converted, copied_only, missing_img = 0, 0, 0, 0

    for json_path in tqdm(json_files):
        base = os.path.basename(json_path)[:-5]  # .json 제거

        # 원본 이미지 찾기 (확장자 가리지 않음)
        src_img_path, src_ext = _find_image_any_ext(gdrive_img_dir, base)
        if src_img_path is None:
            print(f"Warning: Image file not found for {json_path}. Skipping.")
            missing_img += 1
            continue

        # 목적지 경로(이미지는 원본 확장자 그대로)
        dst_img_path   = os.path.join(local_dest_dir, 'images', set_type, base + src_ext)
        dst_label_path = os.path.join(local_dest_dir, 'labels', set_type, base + '.txt')

        # resume/overwrite 처리
        if not overwrite and resume and os.path.exists(dst_img_path) and os.path.exists(dst_label_path):
            skipped += 1
            continue

        # 1) 라벨 생성(필요할 때만)
        if overwrite or not os.path.exists(dst_label_path):
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            data_key = list(data.keys())[0]
            ann = data[data_key]

            # 이미지 크기 (없으면 512 기본)
            W = int(ann.get('file_attributes', {}).get('img_width', 512))
            H = int(ann.get('file_attributes', {}).get('img_height', 512))

            yolo_lines = []
            for r in ann['regions']:
                a = r['shape_attributes']
                x, y, w, h = a['x'], a['y'], a['width'], a['height']
                xc = (x + w/2) / W
                yc = (y + h/2) / H
                nw = w / W
                nh = h / H
                yolo_lines.append(f"0 {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")

            with open(dst_label_path, 'w') as f:
                f.write('\n'.join(yolo_lines))
            converted += 1

        # 2) 이미지 복사(필요할 때만)
        if overwrite or not os.path.exists(dst_img_path):
            shutil.copy2(src_img_path, dst_img_path)
            # 라벨은 있었고 이미지만 없었을 수도 있으므로 통계 분리
            if converted == 0:
                copied_only += 1

    print(f"[{set_type}] done. "
          f"skipped(이미 존재): {skipped}, "
          f"labels_written: {converted}, "
          f"images_only_copied: {copied_only}, "
          f"missing_image: {missing_img}")

# --- 실행 ---
setup_dataset_structure(DATASET_ROOT_LOCAL)

# 학습/검증 데이터 재실행 (이미 처리된 항목은 자동 스킵)
convert_and_copy_data(TRAIN_JSON_DIR_GDRIVE, TRAIN_IMG_DIR_GDRIVE, DATASET_ROOT_LOCAL, 'train', resume=True, overwrite=False)
convert_and_copy_data(VAL_JSON_DIR_GDRIVE,   VAL_IMG_DIR_GDRIVE,   DATASET_ROOT_LOCAL, 'val',   resume=True, overwrite=False)

print("\nData preparation and conversion complete!")

In [ ]:
# ==============================================================================
# 3. 데이터셋 YAML 파일 생성
# ==============================================================================
# YAML 파일에 들어갈 내용 정의
dataset_yaml_config = {
    'path': DATASET_ROOT_LOCAL,
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'chimney'
    }
}

# YAML 파일로 저장
YAML_PATH = os.path.join(DATASET_ROOT_LOCAL, 'chimney_dataset.yaml')
with open(YAML_PATH, 'w') as f:
    yaml.dump(dataset_yaml_config, f, sort_keys=False)

print(f"Dataset YAML file successfully created at: {YAML_PATH}")
print("\n--- YAML 파일 내용 ---")
with open(YAML_PATH, 'r') as f:
    print(f.read())

Dataset YAML file successfully created at: /content/drive/MyDrive/Colab Notebooks/chimney_dataset.yaml

--- YAML 파일 내용 ---
path: /content/drive/MyDrive/Colab Notebooks
train: images/train
val: images/val
names:
  0: chimney



In [ ]:
# ======================================================================
# 4. YOLOv8 Fine-tuning 실행
# ======================================================================
from ultralytics import YOLO
import os

# 기본 설정값
MODEL_NAME = 'yolov8m.pt'
EPOCHS = 100
BATCH = 24
IMG_SIZE = 1024

# 결과 저장 경로 (환경에 맞게 수정, 다음 셀의 시각화에서도 재사용)
RESULTS_DIR = 'drive/MyDrive/DataCreatorCamp/YOLOv8_Results'

# 사전 학습된 YOLO 모델 로드
model = YOLO(MODEL_NAME)
print(f"Start fine-tuning with model: {MODEL_NAME}")

# 실험 이름 자동 생성
model_suffix = os.path.splitext(MODEL_NAME)[0].replace("yolov8", "")
exp_name = f"chimney_detection_{model_suffix}_e{EPOCHS}_b{BATCH}"

# 모델 Fine-tuning 실행
results = model.train(
    # --- 기본 설정 ---
    data=YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=20,

    # --- 프로젝트 및 결과 저장 설정 ---
    project=RESULTS_DIR,
    name=exp_name,   # 자동 생성된 이름 사용

    # --- 데이터 증강 파라미터 ---
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    flipud=0.3,
    mosaic=0.8,
    mixup=0.1,

    # --- 추가 옵션 ---
    seed=42,
    workers=8
)

print("\nFine-tuning has been completed!")
print(f"Results are saved in: {results.save_dir}")

In [ ]:
# ==============================================================================
# 5. 시각화: 2x5 패널(손실/정밀도/재현율/mAP) + Confusion Matrix
# ==============================================================================
import os, pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
from IPython.display import display, Image as IPyImage

# 이전 셀에서 설정한 결과 저장 경로 재사용
PROJECT_DIR = Path(RESULTS_DIR)

def find_latest_run(project_dir: Path) -> Path:
    runs = [p for p in project_dir.glob('*') if p.is_dir()]
    if not runs:
        raise FileNotFoundError(f'No run directories found under: {project_dir}')
    runs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return runs[0]

# 이전 셀의 results.save_dir 우선 사용
try:
    RESULTS_DIR_RUN = Path(results.save_dir)
except NameError:
    RESULTS_DIR_RUN = find_latest_run(PROJECT_DIR)

print(f"[INFO] Using results directory: {RESULTS_DIR_RUN}")

# results.csv 로드
csv_path = RESULTS_DIR_RUN / 'results.csv'
if not csv_path.exists():
    raise FileNotFoundError(f'results.csv not found at: {csv_path}')
df = pd.read_csv(csv_path)
epoch = df.index + 1  # 시각화용 1부터 시작

# 유틸: 컬럼 탐색/스무딩
def find_first_col(candidates):
    """후보 리스트 중 실제 존재하는 첫 컬럼명을 반환, 없으면 None"""
    for cand in candidates:
        # 정확 일치 먼저
        if cand in df.columns:
            return cand
    # 부분일치(버전별 표기 차이 보완)
    for cand in candidates:
        for col in df.columns:
            if cand in col:
                return col
    return None

def smooth(series, window=7):
    """간단 이동평균(odd window 권장)"""
    if window <= 1:
        return series
    return series.rolling(window=window, min_periods=1, center=True).mean()

# 2x5 패널로 그릴 항목 정의(후보명은 버전 호환용)
panels = [
    # 1행
    ("train/box_loss",  ["train/box_loss"]),
    ("train/cls_loss",  ["train/cls_loss"]),
    ("train/dfl_loss",  ["train/dfl_loss"]),
    ("metrics/precision(B)", ["metrics/precision(B)", "metrics/precision"]),
    ("metrics/recall(B)",    ["metrics/recall(B)", "metrics/recall"]),
    # 2행
    ("val/box_loss",    ["val/box_loss"]),
    ("val/cls_loss",    ["val/cls_loss"]),
    ("val/dfl_loss",    ["val/dfl_loss"]),
    ("metrics/mAP50(B)",     ["metrics/mAP50(B)", "metrics/mAP50"]),
    ("metrics/mAP50-95(B)",  ["metrics/mAP50-95(B)", "metrics/mAP@0.50:0.95", "metrics/mAP50-95"]),
]

# 패널 그리기
fig, axes = plt.subplots(2, 5, figsize=(18, 7), constrained_layout=True)

for i, (title, candidates) in enumerate(panels):
    r, c = divmod(i, 5)
    ax = axes[r, c]
    col = find_first_col(candidates)
    if col is None:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center', fontsize=12)
        ax.set_title(title)
        ax.set_xlabel('Epoch'); ax.set_ylabel('')
        ax.grid(True, alpha=0.3)
        continue

    y = df[col]
    y_s = smooth(y, window=7)

    ax.plot(epoch, y, marker='o', markersize=2, linewidth=1.2, label='results')
    ax.plot(epoch, y_s, linestyle=':', linewidth=1.5, label='smooth')
    ax.set_title(col)  # 실제 컬럼명 표기(버전별 표기 차이 그대로 노출)
    ax.set_xlabel('Epoch')
    # y축 라벨 간단화
    if 'loss' in col:
        ax.set_ylabel('Loss')
    elif 'precision' in col.lower():
        ax.set_ylabel('Precision')
    elif 'recall' in col.lower():
        ax.set_ylabel('Recall')
    elif 'mAP' in col:
        ax.set_ylabel('mAP')
    else:
        ax.set_ylabel('')
    ax.grid(True, alpha=0.3)
    # 좌상단 첫 패널에만 범례를 두면 시인성↑
    if i == 0:
        ax.legend(loc='best', fontsize=9)

panel_path = RESULTS_DIR_RUN / 'metrics_panel.png'
fig.suptitle('Training/Validation Metrics', fontsize=14)
fig.savefig(panel_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"[INFO] Saved panel plot to: {panel_path}")

# Confusion Matrix 생성/표시
weights_best = RESULTS_DIR_RUN / 'weights' / 'best.pt'
if not weights_best.exists():
    raise FileNotFoundError(f'best.pt not found at: {weights_best}')

# 이전 셀 변수 재사용(없으면 기본값)
try:
    DATA_YAML = YAML_PATH
except NameError:
    DATA_YAML = None
if DATA_YAML is None:
    raise ValueError("DATA YAML 경로를 찾지 못했습니다. 앞 셀의 YAML_PATH 변수를 그대로 유지하고 실행해주세요.")

try:
    IMG_SIZE_TO_USE = IMG_SIZE
except NameError:
    IMG_SIZE_TO_USE = 1024

# best.pt로 val 평가(plots=True → confusion_matrix.png 생성)
model = YOLO(str(weights_best))
val_results = model.val(
    data=DATA_YAML,
    imgsz=IMG_SIZE_TO_USE,
    plots=True,
    project=str(PROJECT_DIR),
    name='val_plots',
    split='val'
)

VAL_DIR = Path(val_results.save_dir)
print(f"[INFO] Validation plots saved to: {VAL_DIR}")

cm_raw = VAL_DIR / 'confusion_matrix.png'
if cm_raw.exists():
    display(IPyImage(filename=str(cm_raw), width=580))
    print(f"[INFO] Confusion Matrix (raw): {cm_raw}")
else:
    print("[WARN] confusion_matrix.png not found. Check validation results.")
